In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_preprocessing import DataPreprocessor

# Initialize preprocessor
preprocessor = DataPreprocessor()
fraud_df, ip_country_df, credit_df = preprocessor.load_data()

# Clean data
credit_df_clean = preprocessor.clean_credit_data()

In [ ]:
# notebooks/02_eda-creditcard.ipynb

print("\n" + "="*50)
print("CREDIT CARD DATA ANALYSIS")
print("="*50)

# Since credit card features are already PCA-transformed, we focus on:
# 1. Amount analysis
# 2. Time analysis
# 3. Class distribution

# Amount analysis
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Amount distribution
axes[0].hist(credit_df_clean['Amount'], bins=50, edgecolor='black', alpha=0.7, log=True)
axes[0].set_title('Transaction Amount Distribution (Log Scale)')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequency (log)')

# Amount by class
fraud_amounts = credit_df_clean[credit_df_clean['Class'] == 1]['Amount']
non_fraud_amounts = credit_df_clean[credit_df_clean['Class'] == 0]['Amount']

axes[1].boxplot([non_fraud_amounts, fraud_amounts], labels=['Non-Fraud', 'Fraud'])
axes[1].set_title('Transaction Amount by Fraud Status')
axes[1].set_ylabel('Amount')

plt.tight_layout()
plt.savefig('reports/figures/eda_plots/credit_amount_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Time analysis
print("\nTime feature analysis:")
print(f"Time range: {credit_df_clean['Time'].min()} to {credit_df_clean['Time'].max()} seconds")

# Convert time to hours
credit_df_clean['Time_hours'] = credit_df_clean['Time'] / 3600

# Fraud rate over time
plt.figure(figsize=(10, 6))
credit_df_clean['Time_bin'] = pd.cut(credit_df_clean['Time_hours'], bins=48)
fraud_rate_by_time = credit_df_clean.groupby('Time_bin')['Class'].mean()

plt.plot(range(len(fraud_rate_by_time)), fraud_rate_by_time.values)
plt.xlabel('Time Bin (48 bins over dataset duration)')
plt.ylabel('Fraud Rate')
plt.title('Fraud Rate Over Time')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reports/figures/eda_plots/credit_time_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Scale the Amount column (important for modeling)
from sklearn.preprocessing import StandardScaler

credit_processed = credit_df_clean.copy()
scaler = StandardScaler()
credit_processed['Amount_scaled'] = scaler.fit_transform(credit_processed[['Amount']])

# Save processed credit data
credit_processed.to_csv('data/processed/creditcard_processed.csv', index=False)
print("\nSaved processed credit data to: data/processed/creditcard_processed.csv")